Section 1 : Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup

Section 2 : Load Dataset

In [3]:
df = pd.read_csv("../data/interim/balanced_dataset.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (7419, 13)


,City,Listing Type,Field,Job Title,Company Name,Salary/Stipend,Salary_Amount,Job Description,Location,Posted Date,Experience Level,Application Link,Real/Fake Job
0,Mumbai,Full-time,Tech,Software Developer,Rao IIT Academy,"₹500,000 - ₹1,200,000",500000.0,"As a Software Developer at Rao IIT Academy, yo...","Mumbai, Maharashtra",2019-05-17T21:02:37Z,Entry Level (0-2 years),https://www.adzuna.in/details/1155088244?utm_m...,Real Job
1,Mumbai,Full-time,Tech,Software Developer,Cere Labs,"₹300,000 - ₹400,000",300000.0,About us Cere Labs is a Mumbai based company w...,"Mumbai, Maharashtra",2026-01-24T11:20:47Z,Entry Level (0-2 years),https://www.adzuna.in/details/5598463972?utm_m...,Real Job
2,Mumbai,Full-time,Tech,Software Developer,Image Media World,"₹500,000 - ₹700,000",500000.0,Job Description: Software Developer Position O...,"Mumbai, Maharashtra",2024-04-03T18:05:16Z,Entry Level (0-2 years),https://www.adzuna.in/details/4634583545?utm_m...,Real Job
3,Mumbai,Full-time,Tech,Software Developer,Ion,Not specified,NaN,"Job Summary - Writes new software, makes modif...","Mumbai, Maharashtra",2026-05-17T06:47:16Z,Entry Level (0-2 years),https://www.adzuna.in/details/5731961124?utm_m...,Real Job
4,Mumbai,Full-time,Tech,Software Developer,Etaash Consultants,"₹300,000 - ₹1,500,000",300000.0,Job opening in IT company for Angular Develope...,"Mumbai, Maharashtra",2021-09-03T01:56:01Z,Entry Level (0-2 years),https://www.adzuna.in/details/2427499903?utm_m...,Real Job


Section 3 : Missing Values Handling

## 3. Missing Value Handling

Missing values can negatively affect machine learning models and NLP pipelines by introducing inconsistencies and reducing data quality.

In this section, missing categorical values are imputed using meaningful defaults, while highly sparse numerical features are removed to improve downstream model performance.

In [4]:
# Missing Value Handling


# Display missing value statistics
missing_summary = (
    df.isnull()
      .sum()
      .to_frame(name="Missing Count")
)

missing_summary["Missing Percentage"] = (
    missing_summary["Missing Count"] / len(df) * 100
).round(2)

display(missing_summary.sort_values(
    by="Missing Percentage",
    ascending=False
))

,Missing Count,Missing Percentage
Salary_Amount,5776,77.85
Company Name,1,0.01
City,0,0.00
Field,0,0.00
Listing Type,0,0.00
Job Title,0,0.00
Salary/Stipend,0,0.00
Job Description,0,0.00
Location,0,0.00
Posted Date,0,0.00


In [5]:
# Fill Missing Values

fill_values = {
    "Company Name": "Unknown Company",
    "Field": "Unknown",
    "Listing Type": "Full-time",
    "Experience Level": "Entry Level",
    "Salary/Stipend": "Not Specified",
    "Location": "Remote",
    "City": "Unknown",
    "Job Description": ""
}

df.fillna(value=fill_values, inplace=True)

In [6]:
# Remove High Missing Feature

HIGH_MISSING_COLUMN = "Salary_Amount"

if HIGH_MISSING_COLUMN in df.columns:
    df.drop(columns=HIGH_MISSING_COLUMN, inplace=True)

print(f"✓ Dropped Column: {HIGH_MISSING_COLUMN}")

✓ Dropped Column: Salary_Amount


In [7]:
# Final Validation

missing_after = (
    df.isnull()
      .sum()
      .to_frame(name="Remaining Missing")
)

display(missing_after)

print(f"\nFinal Dataset Shape : {df.shape}")

,Remaining Missing
City,0
Listing Type,0
Field,0
Job Title,0
Company Name,0
Salary/Stipend,0
Job Description,0
Location,0
Posted Date,0
Experience Level,0



Final Dataset Shape : (7419, 12)


Section 4 : HTML Cleaning

## 4. HTML Tag Removal

The job descriptions collected from multiple online sources contain HTML tags such as `<p>`, `<div>`, `<ul>`, and `<li>`, which do not contribute meaningful semantic information for machine learning models.

In this section, all HTML elements are removed while preserving the textual content, resulting in clean descriptions suitable for NLP preprocessing.

In [9]:
# HTML Tag Removal

from bs4 import BeautifulSoup

def remove_html_tags(text: str) -> str:
    """
    Remove HTML tags while preserving textual content.

    Parameters
    ----------
    text : str
        Raw HTML text.

    Returns
    -------
    str
        Clean plain text.
    """
    if pd.isna(text):
        return ""

    return BeautifulSoup(str(text), "html.parser").get_text(separator=" ")


# Apply HTML cleaning
df["Job Description"] = df["Job Description"].apply(remove_html_tags)

print("✓ HTML tags removed.")

✓ HTML tags removed.


In [10]:
# Validate HTML Removal

print(df["Job Description"].sample(5))

6718    The Call Center Representative I will provide ...
4048    Review and analyze requirements, technical des...
5722    Role: Product Manager Location: Noida Experien...
5092    Key Responsibilities: Product Owner is the own...
4701    About Bonito Designs Our aim, We are the desig...
Name: Job Description, dtype: object


Section 5 : URL Cleaning

## 5. URL and Email Address Removal

Job descriptions collected from online sources often contain hyperlinks, website URLs, email addresses, and embedded references that do not contribute meaningful information for fraud detection.

In this section, URLs and email addresses are removed to reduce noise and improve the quality of textual features for downstream NLP tasks.

In [11]:
# SECTION 5: URL & Email Removal

import re

def remove_urls_and_emails(text: str) -> str:
    """
    Remove URLs and email addresses from text.

    Parameters
    ----------
    text : str
        Input text.

    Returns
    -------
    str
        Cleaned text.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove Email Addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    return text


# Apply cleaning
df["Job Description"] = df["Job Description"].apply(remove_urls_and_emails)

print("✓ URLs and email addresses removed successfully.")

✓ URLs and email addresses removed successfully.


In [12]:
# Display random cleaned descriptions

df["Job Description"].sample(5)

2375    We are looking for a strong developer to work ...
747     Title: Client Services Intern Role Overview Yo...
5361    Job Description: JOB SUMMARY : General Manager...
388     Media Sales Intern - Mumbai We are seeking a d...
3538    Hiring: Office Admin / Sales Executive Locatio...
Name: Job Description, dtype: object

Section 6 : Text Normalization

## 6. Text Normalization

Raw textual data often contains inconsistent capitalization, punctuation marks, special symbols, HTML entities, and unnecessary whitespace that introduce noise into machine learning models.

This section performs text normalization by converting text to lowercase, decoding HTML entities, removing special characters and punctuation, normalizing whitespace, and producing standardized textual data suitable for NLP feature extraction.

In [13]:
# Text Normalization

import re
import html
import unicodedata

def normalize_text(text: str) -> str:
    """
    Normalize textual data for NLP.

    Steps:
    1. Decode HTML entities
    2. Convert to lowercase
    3. Normalize unicode characters
    4. Remove punctuation & special symbols
    5. Remove digits
    6. Remove extra whitespaces
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Lowercase
    text = text.lower()

    # Unicode normalization
    text = unicodedata.normalize("NFKD", text)

    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Apply normalization

df["Job Description"] = df["Job Description"].apply(normalize_text)

print("✓ Text normalization completed successfully.")

✓ Text normalization completed successfully.


In [14]:
df["Job Description"].sample(5)

1061    job description about the company newton schoo...
5598    role description this is a full time on site i...
4136    we are seeking a detail oriented office accoun...
6608    corporate overview aker solutions is a global ...
1245    about position we are conducting an in person ...
Name: Job Description, dtype: object

Section 7 : Create Combined Text

## 7. Combined Text Generation

Machine learning models often perform better when relevant textual attributes are combined into a single unified feature.

In this section, important textual attributes such as Job Title, Company Name, Field, Experience Level, and Job Description are concatenated into a single feature named **combined_text**.

This unified textual representation will be used for feature extraction using TF-IDF Vectorization in the subsequent notebook.

In [15]:
#Combined Text Generation

# Fill any remaining missing values in text columns

text_columns = [
    "Job Title",
    "Company Name",
    "Field",
    "Experience Level",
    "Job Description"
]

df[text_columns] = df[text_columns].fillna("")

# Create unified NLP feature

df["combined_text"] = (
    df["Job Title"].astype(str)
    + " "
    + df["Company Name"].astype(str)
    + " "
    + df["Field"].astype(str)
    + " "
    + df["Experience Level"].astype(str)
    + " "
    + df["Job Description"].astype(str)
)

print("✓ Combined text feature created successfully.")

✓ Combined text feature created successfully.


In [16]:
# Display sample combined text

df[["combined_text"]].sample(5)

,combined_text
4178,Professor/Associate & Assistant Professor Myci...
4612,Devops Engineer Genetic Callnet Tech Entry Lev...
2615,Field Sales Executive – Freshers/Experienced |...
5499,VMC Operator Navrang HR Solutions Internship E...
4901,Creative Content Strategist - Social Media Dig...


In [17]:
print(df["combined_text"].isnull().sum())

0


## 7.1 Final Combined Text Normalization

After concatenating multiple textual attributes into a single feature, a final normalization step is applied to ensure consistency across the complete text representation.

This includes:
- Lowercase conversion
- HTML entity decoding
- Unicode normalization
- Removal of special characters and punctuation
- Removal of extra whitespaces

The resulting `combined_text` feature will serve as the primary input for TF-IDF vectorization and downstream machine learning models.

In [18]:
# Final Combined Text Normalization

import re
import html
import unicodedata

def clean_combined_text(text: str) -> str:
    """
    Final cleaning pipeline for combined NLP text.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Lowercase
    text = text.lower()

    # Unicode normalization
    text = unicodedata.normalize("NFKD", text)

    # Remove URLs (safety check)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Keep only alphabets and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# Apply cleaning pipeline
df["combined_text"] = df["combined_text"].apply(clean_combined_text)

print("Combined text normalized successfully.")

Combined text normalized successfully.


In [19]:
# Random samples

df["combined_text"].sample(5)

969     management trainee marketing vegetable seeds c...
5201    php developer suyog electricals ltd tech entry...
3064    engineer ii embedded software development cope...
4755    credit manager icici bank limited non tech ent...
7278    assistant accountant immediate start technova ...
Name: combined_text, dtype: object

In [20]:
print("Null Values :", df["combined_text"].isnull().sum())

print("Empty Strings :", (df["combined_text"] == "").sum())

Null Values : 0
Empty Strings : 0


Section 8 : Label Encoding

## 8. Label Encoding

Machine learning algorithms require numerical target labels instead of categorical string values.

In this section, the target variable **Real/Fake Job** is transformed into a binary numerical representation:

- **Real Job → 0**
- **Fake Job → 1**

The encoded label will be used as the target variable for supervised machine learning models in subsequent notebooks.

In [21]:
# Label Encoding

# Define label mapping
LABEL_MAPPING = {
    "Real Job": 0,
    "Fake Job": 1
}

# Create encoded target column
df["label"] = df["Real/Fake Job"].map(LABEL_MAPPING)

print("✓ Label encoding completed successfully.")

✓ Label encoding completed successfully.


In [26]:
# Check label distribution

print(df["label"].value_counts())

print()

print(df[["Real/Fake Job", "label"]].head(10))

label
0    6532
1     887
Name: count, dtype: int64

  Real/Fake Job  label
0      Real Job      0
1      Real Job      0
2      Real Job      0
3      Real Job      0
4      Real Job      0
5      Real Job      0
6      Real Job      0
7      Real Job      0
8      Real Job      0
9      Real Job      0


In [27]:
assert set(df["label"].unique()) == {0, 1}, "Label Encoding Failed!"

print("Target variable validated.")

Target variable validated.


Section 9 : Save Processed dataset

## 9. Export Processed Dataset

After completing the preprocessing pipeline, the cleaned and normalized dataset is exported for downstream feature engineering and machine learning tasks.

The exported dataset contains:
- Clean textual features
- Combined NLP text representation
- Encoded target labels
- Standardized categorical attributes

This processed dataset will serve as the primary input for feature engineering and model training.

In [29]:
# Export Processed Dataset

from pathlib import Path

# Create output directory if it does not exist
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Output file path
output_path = output_dir / "processed_dataset.csv"

# Export dataset
df.to_csv(output_path, index=False)

print("DATASET SAVED")
print(f"Location : {output_path}")
print(f"Shape    : {df.shape}")

DATASET SAVED
Location : ..\data\processed\processed_dataset.csv
Shape    : (7419, 14)


In [30]:
# Final Dataset Validation

print(df.info())

print("\n")

print(df.head())

print("\n")

print(df["label"].value_counts())

print("\n")

print(df["combined_text"].sample(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7419 entries, 0 to 7418
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   City              7419 non-null   object
 1   Listing Type      7419 non-null   object
 2   Field             7419 non-null   object
 3   Job Title         7419 non-null   object
 4   Company Name      7419 non-null   object
 5   Salary/Stipend    7419 non-null   object
 6   Job Description   7419 non-null   object
 7   Location          7419 non-null   object
 8   Posted Date       7419 non-null   object
 9   Experience Level  7419 non-null   object
 10  Application Link  7419 non-null   object
 11  Real/Fake Job     7419 non-null   object
 12  combined_text     7419 non-null   object
 13  label             7419 non-null   int64 
dtypes: int64(1), object(13)
memory usage: 811.6+ KB
None


     City Listing Type Field           Job Title        Company Name  \
0  Mumbai    Full-time  T